# 06 - Dashboard Data

## Power BI Dashboard Data Preparation

This notebook creates Power BI-ready analytical views from the cleaned Silver layer.

**Input tables**
- `sales_silver`
- `stores_silver`
- `products_silver`
- `inventory_silver`
- `suppliers_silver`
- `purchase_orders_silver`

The views are designed for Power BI dashboards covering Sales, Inventory, and Supplier Performance.


## 1. Sales — Monthly Store Performance

Use this dataset for monthly revenue trends, transaction count, quantity sold, and average transaction value by store.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_sales_monthly AS
SELECT
    s.store_id,
    s.store_name,
    DATE_FORMAT(sa.sale_date, 'yyyy-MM') AS sales_month,
    SUM(sa.quantity_sold) AS quantity_sold,
    COUNT(DISTINCT sa.transaction_id) AS transaction_count,
    ROUND(SUM(sa.total_amount), 2) AS total_revenue,
    ROUND(
        SUM(sa.total_amount) / COUNT(DISTINCT sa.transaction_id),
        2
    ) AS avg_transaction_value
FROM sales_silver sa
INNER JOIN stores_silver s
    ON sa.store_id = s.store_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    s.store_id,
    s.store_name,
    DATE_FORMAT(sa.sale_date, 'yyyy-MM');


In [0]:
%sql

SELECT *
FROM dashboard_sales_monthly
ORDER BY store_id, sales_month;


## 2. Sales — Store Performance

One row per store for the main 2024 KPIs and rankings.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_store_performance AS
WITH store_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        SUM(sa.quantity_sold) AS quantity_sold,
        COUNT(DISTINCT sa.transaction_id) AS transaction_count,
        ROUND(SUM(sa.total_amount), 2) AS total_revenue
    FROM sales_silver sa
    INNER JOIN stores_silver s
        ON sa.store_id = s.store_id
    WHERE YEAR(sa.sale_date) = 2024
    GROUP BY s.store_id, s.store_name
)
SELECT
    store_id,
    store_name,
    quantity_sold,
    transaction_count,
    total_revenue,
    ROUND(total_revenue / transaction_count, 2) AS avg_transaction_value,
    DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    DENSE_RANK() OVER (ORDER BY transaction_count DESC) AS transaction_rank
FROM store_metrics;


In [0]:
%sql

SELECT *
FROM dashboard_store_performance
ORDER BY revenue_rank;


## 3. Sales — Product Performance

Product and category metrics for Top-N charts and category comparisons.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_product_sales AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(sa.quantity_sold) AS quantity_sold,
    ROUND(SUM(sa.total_amount), 2) AS total_revenue,
    COUNT(DISTINCT sa.transaction_id) AS transaction_count
FROM sales_silver sa
INNER JOIN products_silver p
    ON sa.product_id = p.product_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY
    p.product_id,
    p.product_name,
    p.category;


In [0]:
%sql

SELECT *
FROM dashboard_product_sales
ORDER BY quantity_sold DESC, total_revenue DESC;


## 4. Inventory — Current Stock

Current inventory by store and product, including stock-status classification.

Business rule:
- `0` = Out of Stock
- `< 50` = Low Stock
- `>= 50` = Normal Stock


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_inventory AS
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    CASE
        WHEN i.quantity_on_hand = 0 THEN 'Out of Stock'
        WHEN i.quantity_on_hand < 50 THEN 'Low Stock'
        ELSE 'Normal Stock'
    END AS stock_status
FROM inventory_silver i
INNER JOIN stores_silver s
    ON i.store_id = s.store_id
INNER JOIN products_silver p
    ON i.product_id = p.product_id;


In [0]:
%sql

SELECT *
FROM dashboard_inventory
ORDER BY quantity_on_hand ASC, store_id, product_id;


## 5. Inventory — Risk Analysis

Combines current stock with 2024 average daily sales.

The median is used as the threshold because the capstone does not specify a numerical threshold for slow-moving or overstock products.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_inventory_risk AS

WITH product_velocity AS (
    SELECT
        product_id,
        SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY product_id
),
velocity_threshold AS (
    SELECT percentile_approx(avg_daily_sales, 0.5) AS median_daily_sales
    FROM product_velocity
),
inventory_threshold AS (
    SELECT percentile_approx(quantity_on_hand, 0.5) AS median_inventory
    FROM inventory_silver
)
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    ROUND(COALESCE(v.avg_daily_sales, 0), 2) AS avg_daily_sales,
    CASE
        WHEN i.quantity_on_hand = 0 THEN 'Out of Stock'
        WHEN i.quantity_on_hand < 50 THEN 'Low Stock'
        ELSE 'Normal Stock'
    END AS stock_status,
    CASE
        WHEN COALESCE(v.avg_daily_sales, 0) > vt.median_daily_sales
             AND i.quantity_on_hand < 50
            THEN 'High Velocity - Low Stock'
        WHEN COALESCE(v.avg_daily_sales, 0) < vt.median_daily_sales
             AND i.quantity_on_hand > it.median_inventory
            THEN 'Slow-Moving - Potential Overstock'
        WHEN COALESCE(v.avg_daily_sales, 0) > vt.median_daily_sales
            THEN 'High Velocity'
        WHEN i.quantity_on_hand > it.median_inventory
            THEN 'Potential Overstock'
        ELSE 'Normal'
    END AS inventory_flag
FROM inventory_silver i
INNER JOIN stores_silver s
    ON i.store_id = s.store_id
INNER JOIN products_silver p
    ON i.product_id = p.product_id
LEFT JOIN product_velocity v
    ON i.product_id = v.product_id
CROSS JOIN velocity_threshold vt
CROSS JOIN inventory_threshold it;


In [0]:
%sql

SELECT *
FROM dashboard_inventory_risk
ORDER BY quantity_on_hand ASC, store_id, product_id;


## 6. Supplier Performance

Supplier metrics based on the fields available in the project data.

**Limitation:** there is no promised/expected delivery date, so a true on-time delivery rate cannot be calculated. We therefore expose completion rate and average delivery days.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_supplier_performance AS
SELECT
    s.supplier_id,
    s.supplier_name,
    COUNT(po.po_id) AS total_orders,
    SUM(
        CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END
    ) AS completed_orders,
    ROUND(
        100.0 * SUM(
            CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END
        ) / COUNT(po.po_id),
        2
    ) AS completion_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN po.delivery_date IS NOT NULL
                THEN DATEDIFF(po.delivery_date, po.order_date)
            END
        ),
        2
    ) AS avg_delivery_days
FROM purchase_orders_silver po
INNER JOIN suppliers_silver s
    ON po.supplier_id = s.supplier_id
GROUP BY s.supplier_id, s.supplier_name;


In [0]:
%sql

SELECT *
FROM dashboard_supplier_performance
ORDER BY completion_rate_pct DESC, avg_delivery_days ASC;


## 7. Reorder Point Preparation

Baseline formula:

`Reorder Point = Average Daily Sales × Average Supplier Lead Time`

This is a baseline recommendation. The project data does not contain demand variability or a service-level target, so safety stock is not calculated here.


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dashboard_reorder_point AS

WITH sales_velocity AS (
    SELECT
        store_id,
        product_id,
        SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY store_id, product_id
),
supplier_lead_time AS (
    SELECT
        supplier_id,
        AVG(DATEDIFF(delivery_date, order_date)) AS avg_lead_time_days
    FROM purchase_orders_silver
    WHERE delivery_date IS NOT NULL
    GROUP BY supplier_id
)
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    p.supplier_id,
    ROUND(COALESCE(v.avg_daily_sales, 0), 2) AS avg_daily_sales,
    ROUND(COALESCE(sl.avg_lead_time_days, 0), 2) AS avg_supplier_lead_time_days,
    ROUND(
        COALESCE(v.avg_daily_sales, 0)
        * COALESCE(sl.avg_lead_time_days, 0),
        2
    ) AS baseline_reorder_point,
    CASE
        WHEN i.quantity_on_hand <=
             COALESCE(v.avg_daily_sales, 0)
             * COALESCE(sl.avg_lead_time_days, 0)
            THEN 'Reorder Recommended'
        ELSE 'Stock Above Reorder Point'
    END AS reorder_status
FROM inventory_silver i
INNER JOIN stores_silver s
    ON i.store_id = s.store_id
INNER JOIN products_silver p
    ON i.product_id = p.product_id
LEFT JOIN sales_velocity v
    ON i.store_id = v.store_id
   AND i.product_id = v.product_id
LEFT JOIN supplier_lead_time sl
    ON p.supplier_id = sl.supplier_id;


In [0]:
%sql

SELECT *
FROM dashboard_reorder_point
ORDER BY store_id, product_id;


## Power BI Consumption

The main Power BI datasets created by this notebook are:

| View | Purpose |
|---|---|
| `dashboard_sales_monthly` | Monthly sales trends |
| `dashboard_store_performance` | Store KPIs and rankings |
| `dashboard_product_sales` | Product/category performance |
| `dashboard_inventory` | Current stock dashboard |
| `dashboard_inventory_risk` | Inventory risk analysis |
| `dashboard_supplier_performance` | Supplier performance |
| `dashboard_reorder_point` | Reorder recommendations |

These are **temporary Databricks views** for dashboard development/testing. If the Power BI connection requires persistent Delta tables, we can materialize the finalized views after validating the results.
